# SOUL AI — G3.1 Colab evaluation

Notebook chạy inference/evaluation cho **frozen world-model checkpoint G3.1** trên dữ liệu Pendulum đã lưu trong repo.

- Checkpoint: chỉ TRAIN (seeds 1001–1005).
- Đánh giá: PILOT đã ghi nhận (seeds 2001–2005).
- Baselines: persistence và mean-delta chỉ fit từ TRAIN.
- Không huấn luyện lại, không ghi đè artifact trong repo, không mở CONFIRM.
- Kết quả luôn được gắn nhãn exploratory; threshold chưa khóa nên không có kết luận PASS/FAIL về chất lượng mô hình.

> Đây là bài kiểm tra world-model dự đoán trạng thái tiếp theo trong Pendulum-v1, không phải chatbot và cũng chưa phải agent tự chơi Minecraft.

In [ ]:
# 1) Clone đúng nhánh Colab, không dùng main thay thế
import os, pathlib, shutil, subprocess, sys

REPO_URL = "https://github.com/nguyenphuck2468-tech/Soul-AI.git"
BRANCH = "colab/g31-pilot-evaluation"
REPO_DIR = pathlib.Path("/content/Soul-AI")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
    check=True,
)
os.chdir(REPO_DIR)
print("Branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

In [ ]:
# 2) Install the repository's pinned dependencies
# The evaluation uses CPU intentionally; a GPU is not required.
%pip install --disable-pip-version-check -r requirements.lock

In [ ]:
# 3) Verify protocol boundaries and the frozen checkpoint provenance
import hashlib, json, torch
from pathlib import Path

subprocess.run([sys.executable, "-m", "scripts.phase0_validate_protocol"], check=True)

checkpoint_path = Path("artifacts/g31/g31_verified_train_checkpoint.pt")
pilot_path = Path("artifacts/g31/pilot.jsonl")
train_path = Path("artifacts/g31/train.jsonl")

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

for path in (checkpoint_path, pilot_path, train_path):
    if not path.is_file():
        raise FileNotFoundError(path)

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
assert checkpoint.get("split") == "TRAIN", checkpoint.get("split")
assert checkpoint.get("data_split") == "TRAIN", checkpoint.get("data_split")
assert sorted(checkpoint.get("train_seeds", [])) == [1001, 1002, 1003, 1004, 1005]
assert checkpoint.get("environment_id") == "Pendulum-v1"

print("Checkpoint:", checkpoint_path)
print("Checkpoint SHA-256:", sha256(checkpoint_path))
print("Checkpoint provenance: TRAIN only, seeds 1001–1005")
print("TRAIN dataset SHA-256:", sha256(train_path))
print("PILOT dataset SHA-256:", sha256(pilot_path))
print("CONFIRM: not loaded")

In [ ]:
# 4) Run the official project evaluator (zero optimizer updates)
from datetime import datetime, timezone

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT = Path("work") / "colab_g31_eval" / run_id
OUT.mkdir(parents=True, exist_ok=False)

cmd = [
    sys.executable, "-m", "soul.g31.score_pilot",
    "--checkpoint", str(checkpoint_path),
    "--predictions", str(OUT / "predictions.jsonl"),
    "--metrics", str(OUT / "metrics.json"),
    "--report", str(OUT / "report.json"),
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("Output:", OUT)

In [ ]:
# 5) Inspect results and enforce the no-CONFIRM/no-training contract
metrics = json.loads((OUT / "metrics.json").read_text(encoding="utf-8"))
report = json.loads((OUT / "report.json").read_text(encoding="utf-8"))

assert report["confirm_accessed"] is False
assert report["optimizer_updates"] == 0
assert report["threshold"] == "TO_LOCK"
assert report["gate"] == "INCONCLUSIVE"
assert metrics["threshold"] == "TO_LOCK"
assert metrics["pilot_scoring"].startswith("EXPLORATORY")

print(json.dumps(report, indent=2, ensure_ascii=False))

def show_aggregate(label, payload):
    print("\n" + label)
    for metric, values in payload.items():
        print(
            f"{metric:16s} mean={values['mean']:.6f} "
            f"median={values['median']:.6f} "
            f"dispersion={values['dispersion']:.6f} "
            f"episodes={values['episode_count']}"
        )

show_aggregate("SOUL G3.1 model", metrics["model_aggregate_equal_episode"])
show_aggregate("Persistence baseline", metrics["persistence_aggregate_equal_episode"])
show_aggregate("TRAIN-only mean-delta baseline", metrics["mean_delta_aggregate_equal_episode"])

print("\nPaired model - persistence:")
print(json.dumps(metrics["paired_model_minus_persistence"], indent=2))
print("\nInterpretation: descriptive PILOT metrics only; threshold remains TO_LOCK.")

In [ ]:
# 6) Plot one recorded PILOT episode: predicted vs actual physical state
import matplotlib.pyplot as plt
import numpy as np

records = [
    json.loads(line)
    for line in (OUT / "predictions.jsonl").read_text(encoding="utf-8").splitlines()
]
episode_ids = sorted({row["episode_id"] for row in records})
selected_episode = episode_ids[0]
episode_rows = sorted(
    [row for row in records if row["episode_id"] == selected_episode],
    key=lambda row: row["timestep"],
)
steps = [row["timestep"] for row in episode_rows]
pred = np.asarray([row["predicted_physical"] for row in episode_rows])
target = np.asarray([row["target_physical"] for row in episode_rows])

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(steps, target[:, 0], label="Actual theta")
axes[0].plot(steps, pred[:, 0], label="Predicted theta")
axes[0].set_ylabel("Angle (rad)")
axes[0].legend()
axes[0].grid(True, alpha=0.25)

axes[1].plot(steps, target[:, 1], label="Actual angular velocity")
axes[1].plot(steps, pred[:, 1], label="Predicted angular velocity")
axes[1].set_xlabel("Timestep")
axes[1].set_ylabel("Angular velocity (rad/s)")
axes[1].legend()
axes[1].grid(True, alpha=0.25)
fig.suptitle(f"G3.1 PILOT episode: {selected_episode}")
plt.tight_layout()
plt.show()

In [ ]:
# 7) Package this run's outputs for download
import shutil
from google.colab import files

archive_base = f"/content/SOUL_AI_G31_Colab_{run_id}"
archive_path = shutil.make_archive(archive_base, "zip", root_dir=OUT)
print("Created:", archive_path)
files.download(archive_path)

## Cách đọc kết quả

- **theta** được so sánh theo sai số góc tuần hoàn; **velocity** theo rad/s.
- Mỗi episode được tính riêng rồi tổng hợp ngang trọng số episode, tránh episode dài lấn át.
- So sánh model với persistence và mean-delta baseline; xem cả trung bình, trung vị và độ phân tán.
- Nếu model kém baseline, đó là kết quả âm cần giữ lại, không được che đi bằng cách đổi seed hoặc sửa threshold sau khi xem kết quả.
- Notebook chỉ đánh giá PILOT đã tồn tại trong repo. Không chạy lệnh thu thập, huấn luyện, khóa threshold, hay bất kỳ lệnh CONFIRM nào.

Tham khảo mã đánh giá gốc: soul/g31/score_pilot.py và soul/g31/evaluator.py.